In [1]:
import os
import numpy as np
import pandas as pd

# -------- (LATITUDE, LONGITUDE, ELEVATION) --------
# USW00014707: (41.3275, -72.0494, 3.0)
# USW00014740: (41.9375, -72.6819, 53.3)
# USW00014758: (41.2639, -72.8872, 0.9)
# USW00054734: (41.3714, -73.4828, 139.3)
# USW00054767: (41.7419, -72.1836, 75.3)
# USW00054788: (41.5097, -72.8278, 31.4)
# USW00094702: (41.1583, -73.1289, 1.5)

stations = ['USW00014707','USW00014740','USW00014758','USW00054734','USW00054767','USW00054788','USW00094702']
cols_1 = ['DATE','HourlyPrecipitation','HourlyDryBulbTemperature','HourlyRelativeHumidity','HourlySeaLevelPressure','HourlyWindSpeed','HourlyWindDirection']
cols_2 = ['time','apcp','air','rhum','prmsl','uwnd','vwnd']

In [ ]:
# LCD data preprocessing
num = 0
LCD_all = []

for year in range(1979, 2025):
    file = f'LCD_{stations[num]}_{year}.csv'
    if not os.path.exists(f'LCD_data/{file}'):
        print(f'{year}: {file} does not exist.')
        continue
    LCD = pd.read_csv(f'LCD_data/{file}', low_memory=False)[cols_1]
    LCD['DATE'] = pd.to_datetime(LCD['DATE'], errors='coerce')
    print(f'{year}: Records = {len(LCD):6,d} | NaN = {LCD.isna().any(axis=1).sum():6,d} | {LCD['DATE'].min()} -> {LCD['DATE'].max()}')
    LCD_all.append(LCD)

LCD_all = pd.concat(LCD_all, ignore_index=True)
LCD_all = LCD_all.sort_values('DATE').reset_index(drop=True)
LCD_all.to_csv(f'LCD_data/LCD_{stations[num]}.csv', index=False)

In [ ]:
# NARR data preprocessing
for station in stations:
    NARR = pd.read_csv(f'NARR_data/{station}.csv', skiprows=1, low_memory=False)[cols_2]

    # UTC -> LST (EST, UTC-5)
    NARR['time'] = pd.to_datetime(NARR['time'], errors='coerce', utc=True).dt.tz_convert('Etc/GMT+5').dt.tz_localize(None)
    
    # Convert to hourly records
    NARR = NARR.set_index('time').reindex(pd.date_range(NARR['time'].min(), NARR['time'].max(), freq='h'))

    # Evenly distribute 3-hour accumulated precipitation
    NARR['apcp'] = (NARR['apcp'] / 3).bfill()

    # Linearly interpolate state variables
    NARR[cols_2[2:]] = NARR[cols_2[2:]].interpolate(method='time')

    # Unit conversion
    NARR['air'] = NARR['air'] - 273.15   # K to degC
    NARR['prmsl'] = NARR['prmsl'] / 100  # Pa to hPa

    # Save data
    NARR = NARR.rename_axis('time').reset_index()
    NARR = NARR.sort_values('time').reset_index(drop=True)
    NARR.to_csv(f'NARR_data/NARR_{station}.csv', index=False)

In [2]:
for station in stations:
    # Read LCD data
    df = pd.read_csv(f'LCD_data/LCD_{station}.csv', low_memory=False)
    df['DATE'] = pd.to_datetime(df['DATE'], errors='coerce')
    df = df[df['DATE'].dt.year >= 2005].copy()

    # Clean columns
    df['HourlyPrecipitation'] = pd.to_numeric(df['HourlyPrecipitation'].replace('T', 0), errors='coerce')
    df['HourlyRelativeHumidity'] = pd.to_numeric(df['HourlyRelativeHumidity'], errors='coerce').mask(lambda x: x > 105)
    df['HourlyWindSpeed'] = pd.to_numeric(df['HourlyWindSpeed'].astype(str).str.rstrip('s'), errors='coerce')
    df['HourlyWindDirection'] = pd.to_numeric(df['HourlyWindDirection'].astype(str).str.rstrip('s'), errors='coerce').mask(lambda x: x == 999)
    for col in ['HourlyDryBulbTemperature', 'HourlySeaLevelPressure']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    
    # Convert wind speed and direction to eastward and northward components
    theta = np.deg2rad(df['HourlyWindDirection'])
    df['WindEast'] = -df['HourlyWindSpeed'] * np.sin(theta)
    df['WindNorth'] = -df['HourlyWindSpeed'] * np.cos(theta)

    # Aggregate records to integer hours
    df['DATE'] = df['DATE'].dt.floor('h')
    df = df.groupby('DATE', as_index=False).agg({
        'HourlyPrecipitation': lambda x: x.sum(min_count=1),
        'HourlyDryBulbTemperature': 'mean',
        'HourlyRelativeHumidity': 'mean',
        'HourlySeaLevelPressure': 'mean',
        'WindEast': 'mean',
        'WindNorth': 'mean'})

    # Create continuous hourly records
    hourly_idx = pd.date_range('2005-01-01 00:00:00', '2024-12-31 23:00:00', freq='h')
    df = df.set_index('DATE').reindex(hourly_idx).rename_axis('DATE').reset_index()
    
    # Read NARR data
    NARR = pd.read_csv(f'NARR_data/NARR_{station}.csv', low_memory=False)
    NARR['time'] = pd.to_datetime(NARR['time'], errors='coerce')
    NARR = NARR.rename(columns=dict(zip(cols_2, df.columns)))

    # Fill missing values with NARR data
    df = df.merge(NARR, on='DATE', how='left', suffixes=('', '_RA'))
    df = df.fillna(df.filter(like='_RA').rename(columns=lambda x: x.replace('_RA', '')))
    df = df[df.columns[~df.columns.str.endswith('_RA')]]
    df.iloc[-2:] = df.iloc[-2:].fillna(df.iloc[-3])
    df['HourlyRelativeHumidity'] = df['HourlyRelativeHumidity'].clip(0, 100)

    # Save processed data
    df = df.sort_values('DATE').reset_index(drop=True)
    df.to_csv(f'{station}.csv', index=False)